# MambaIR — Google Colab eğitim not defteri

Bu not defterini Colab’a yükleyip sırayla çalıştırın.

1. **Runtime → Change runtime type → GPU** (T4/L4 vb.) seçin.
2. Kendi GitHub çatallamanızı (`fork`) kullanacaksanız aşağıdaki `REPO_URL` değerini güncelleyin.
3. Eğitim verilerini Google Drive’a koyup hücredeki yolları kendi klasör yapınıza göre düzenleyin.
### Colab'a nasıl bağlanırım?

- **`Colab: False` görüyorsanız** not defteri şu an **yerelde** (Cursor, VS Code, Jupyter) çalışıyordur; bu **beklenen** davranıştır.
- **`Colab: True` için** tarayıcıda [colab.research.google.com](https://colab.research.google.com) açın → **File → Upload notebook** ile bu `.ipynb` dosyasını yükleyin (veya repoyu GitHub'a koyup Colab'da **File → Open notebook → GitHub** ile açın) → **Runtime → Change runtime type → GPU** seçin → hücreleri **Colab sayfasında** çalıştırın.


## 1. Colab ortamı

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Colab:", IN_COLAB)
if not IN_COLAB:
    print(
        "→ Yerelde çalışıyorsunuz; True görmek için not defterini https://colab.research.google.com üzerinde açın."
    )


In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime türünde GPU seçili değil."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 2. Ayarlar

- `REPO_URL`: Clone edilecek repo (geliştirmeleriniz kendi fork’ınızdaysa onu yazın).
- `REPO_ROOT`: Kodun duracağı klasör (Colab’da genelde `/content/MambaIR`).
- Veri yolları: DIV2K / DF2K yapınıza göre HR ve LR klasörleri.

In [ ]:
from pathlib import Path

# --- GitHub ---
REPO_URL = "https://github.com/csguoh/MambaIR.git"  # Kendi fork: https://github.com/KULLANICI/MambaIR.git
REPO_BRANCH = "main"  # veya çalıştığınız dal

REPO_ROOT = Path("/content/MambaIR")

# --- Google Drive (veri setleri) ---
MOUNT_DRIVE = True  # Veriyi Drive’dan okuyacaksanız True

# Örnek: Drive’da "MambaIR_data/DIV2K_train_HR" gibi yapı
DRIVE_DATA = Path("/content/drive/MyDrive/MambaIR_data")
GT_TRAIN = DRIVE_DATA / "DIV2K_train_HR"
LQ_TRAIN = DRIVE_DATA / "DIV2K_train_LR_bicubic" / "X2"  # x2 için; x3/x4 ise X3, X4

# Doğrulama (ör. Set14); yoksa geçici olarak train alt klasörü vermeyin—küçük bir val seti hazırlayın
GT_VAL = DRIVE_DATA / "Set14" / "HR"
LQ_VAL = DRIVE_DATA / "Set14" / "LR_bicubic" / "X2"

## 3. Google Drive bağlama

In [ ]:
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Drive atlanıyor (Colab değil veya MOUNT_DRIVE=False).")

## 4. Repoyu indirme / güncelleme

In [ ]:
import subprocess
import sys


def sh(*args, cwd=None):
    print("$", " ".join(args))
    subprocess.check_call(args, cwd=cwd)


if not REPO_ROOT.is_dir():
    sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT))
else:
    sh("git", "-C", str(REPO_ROOT), "fetch", "--all")
    sh("git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH)
    sh("git", "-C", str(REPO_ROOT), "pull", "origin", REPO_BRANCH)

print("Repo:", REPO_ROOT)

## 5. Python bağımlılıkları ve `basicsr` kurulumu

Colab’daki PyTorch sürümüne uyum için `mamba-ssm` / `causal-conv1d` bazen ek deneme gerektirir; hata alırsanız [mamba](https://github.com/state-spaces/mamba/releases) ve [causal-conv1d](https://github.com/Dao-AILab/causal-conv1d/releases) üzerindeki tekerlek (wheel) sürümlerini PyTorch + CUDA sürümünüze göre seçebilirsiniz.

`setup.py` içindeki `requirements.txt` conda formatında olduğu için kurulum **`--no-deps`** ile yapılıyor; aşağıdaki paketler elle ekleniyor.

In [ ]:
PIP_DEPS = [
    "addict",
    "einops",
    "future",
    "lmdb",
    "numpy",
    "opencv-python",
    "pillow",
    "pyyaml",
    "requests",
    "scikit-image",
    "scipy",
    "tensorboard",
    "timm",
    "tqdm",
    "yapf",
    "packaging",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_DEPS])

# Mamba çekirdeği (Colab GPU ile uyumlu sürüm pip çözmeye çalışır)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "causal-conv1d", "mamba-ssm"])

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT), "--no-deps"]
)

print("Kurulum tamam.")

## 6. Kurulum doğrulama

In [ ]:
import basicsr

print("basicsr import OK")

## 7. Colab için opsiyon dosyası (Drive yolları + tek GPU)

Orijinal `.yml` dosyalarındaki mutlak yollar yerine yukarıda tanımlı `GT_TRAIN` vb. kullanılır. Batch ve worker değerleri Colab belleğine göre düşürülmüştür; VRAM yetmezse `BATCH_SIZE` değerini azaltın.

In [ ]:
import yaml

BASE_YML = REPO_ROOT / "options/train/mambairv2/train_MambaIRv2_lightSR_x2.yml"
COLAB_YML = REPO_ROOT / "options/train/colab_generated_lightSR_x2.yml"

BATCH_SIZE = 4  # T4’te gerekirse 2 yapın
NUM_WORKERS = 2

assert GT_TRAIN.is_dir(), f"Bulunamadı: {GT_TRAIN}"
assert LQ_TRAIN.is_dir(), f"Bulunamadı: {LQ_TRAIN}"
assert GT_VAL.is_dir(), f"Bulunamadı: {GT_VAL}"
assert LQ_VAL.is_dir(), f"Bulunamadı: {LQ_VAL}"

with open(BASE_YML, "r", encoding="utf-8") as f:
    opt = yaml.load(f, Loader=yaml.FullLoader)

opt["name"] = opt.get("name", "colab") + "_colab"
opt["num_gpu"] = 1
opt["datasets"]["train"]["dataroot_gt"] = [str(GT_TRAIN)]
opt["datasets"]["train"]["dataroot_lq"] = [str(LQ_TRAIN)]
opt["datasets"]["train"]["batch_size_per_gpu"] = BATCH_SIZE
opt["datasets"]["train"]["num_worker_per_gpu"] = NUM_WORKERS

for k in opt["datasets"]:
    if k.startswith("val"):
        opt["datasets"][k]["dataroot_gt"] = str(GT_VAL)
        opt["datasets"][k]["dataroot_lq"] = str(LQ_VAL)

with open(COLAB_YML, "w", encoding="utf-8") as f:
    yaml.dump(opt, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("Yazıldı:", COLAB_YML)

## 8. Eğitimi başlatma

Log ve ağırlıklar `experiments/` altında oluşur (`REPO_ROOT` içinde). Oturum kapanınca `/content` silinir; çıktıları **Drive’a kopyalayın** veya repo kökünü Drive üzerinde tutacak şekilde `git clone` hedefini değiştirin.

In [ ]:
cmd = [
    sys.executable,
    str(REPO_ROOT / "basicsr/train.py"),
    "-opt",
    str(COLAB_YML),
    "--launcher",
    "none",
]

print(" ".join(cmd))
# Eğitimi başlat:
subprocess.check_call(cmd, cwd=str(REPO_ROOT))